In [19]:
# Sampling the data for training and out-of-time testing

import pandas as pd
import numpy as np

df_raw = pd.read_csv("data/abt_churn.csv")

#defining the df we are using for the model
filtro = df_raw["dtRef"] != '2025-04-01'
filtro_1 = df_raw["dtRef"] == '2025-04-01'

df_train = df_raw[filtro].copy().reset_index(drop=True)
oot = df_raw[filtro_1].copy().reset_index(drop=True) # out of time

X = df_train.drop(columns=["flagChurn", "dtRef", "idUsuario"]).copy()
y = df_train["flagChurn"].copy()


In [20]:
# Train-test split

from sklearn import model_selection

X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(y_train.mean(), y_test.mean(), oot["flagChurn"].mean())

0.46894559460760715 0.4687199230028874 0.504950495049505


In [35]:
# importing libraries

from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier

In [36]:
#models

preprocessor = StandardScaler()

pipe_log = Pipeline([
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

pipe_rf = Pipeline([
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

pipe_nb = Pipeline([
    ("model", GaussianNB())
])

# Parameters grids

param_grid_log = {
    "model__C": [0.09, 0.1, 0.11],
    "model__penalty": ["l2"]  # scikit-learn logistic regression defaults
}

param_grid_rf = {
    "model__n_estimators": [150, 200, 250],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5]
}

param_grid_nb = {
    # GaussianNB has only one major hyperparam (var_smoothing)
    "model__var_smoothing": [1e-5, 1e-4, 1e-3]
}

# Grid Search with Cross-Validation

grids = {
    "Logistic Regression": GridSearchCV(pipe_log, param_grid_log, cv=5, scoring="roc_auc", n_jobs=-1),
    "Random Forest": GridSearchCV(pipe_rf, param_grid_rf, cv=5, scoring="roc_auc", n_jobs=-1),
    "Naive Bayes": GridSearchCV(pipe_nb, param_grid_nb, cv=5, scoring="roc_auc", n_jobs=-1)
}

results = {}

for name, grid in grids.items():
    print(f"\n===== {name} =====")
    grid.fit(X_train, y_train)
    y_pred = grid.predict(X_test)
    y_prob = grid.predict_proba(X_test)[:, 1]

    print("Best Params:", grid.best_params_)
    print(classification_report(y_test, y_pred, digits=3))
    print("ROC-AUC:", roc_auc_score(y_test, y_prob))
    
    results[name] = {
        "best_params": grid.best_params_,
        "roc_auc": roc_auc_score(y_test, y_prob)
    }


===== Logistic Regression =====
Best Params: {'model__C': 0.11, 'model__penalty': 'l2'}
              precision    recall  f1-score   support

           0      0.808     0.694     0.747       552
           1      0.701     0.813     0.753       487

    accuracy                          0.750      1039
   macro avg      0.754     0.753     0.750      1039
weighted avg      0.758     0.750     0.750      1039

ROC-AUC: 0.8289326845817337

===== Random Forest =====
Best Params: {'model__max_depth': 10, 'model__min_samples_split': 5, 'model__n_estimators': 200}
              precision    recall  f1-score   support

           0      0.795     0.716     0.753       552
           1      0.710     0.791     0.748       487

    accuracy                          0.751      1039
   macro avg      0.753     0.753     0.751      1039
weighted avg      0.755     0.751     0.751      1039

ROC-AUC: 0.822765080498765

===== Naive Bayes =====
Best Params: {'model__var_smoothing': 1e-05}
        